# Phase 5: Generate summary plots

Phase 5 reads only Phase 4 release tables, so rerunning figures does not repeat PubMed fetches or Jev requests. It reproduces the Version 1 emergence view and concept-profile overview, plus accepted evidence, publication-year, and taxonomy summaries. Outputs and source paths are recorded in `data/05_plots/release/plot_creation_metadata.json`. Set `run_phase: true` in `configs/notebook_parameters.yaml` to generate the figures.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v2").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v2 import notebook_utils as nbutils
from pathogen_atlas_v2.common.io import resolve_path
NOTEBOOK_CONFIG_PATH = "configs/notebook_parameters.yaml"
PARAMS = nbutils.notebook_phase_parameters("phase5_plot_creation", NOTEBOOK_CONFIG_PATH)

import importlib
phase5_module = importlib.import_module("pathogen_atlas_v2.pipelines.05_make_plots")
phase5_preflight = phase5_module.phase5_preflight
build_phase5_plots = phase5_module.build_phase5_plots
from pathogen_atlas_v2.plots.emergence import plot_emergence_timeline
from pathogen_atlas_v2.plots.embeddings import plot_species_feature_space
from pathogen_atlas_v2.plots.taxonomy import plot_taxonomy_composition
from pathogen_atlas_v2.plots.dashboards import (
    plot_species_evidence_counts,
    plot_feature_prevalence,
    plot_annual_evidence_counts,
)


## Inputs and figures

- `species_summary.parquet` → accepted-evidence counts by species and taxonomy ranks.
- `emergence.parquet` → first accepted evidence year versus year reaching the publication target.
- `species_feature_summary.parquet` → prevalence of the 73 V1 expert concepts.
- `evidence_records.parquet` → unique accepted PMIDs by publication year.
- `species_feature_matrix.parquet` → deterministic two-component PCA over species feature profiles, analogous to the V1 concept embeddings.

The functions live in `pathogen_atlas_v2/plots/`. Feature projection is descriptive and does not assign clusters or pathogenicity labels.

In [ ]:
preflight = phase5_preflight(PARAMS)
display(preflight)
display({
    "species_evidence_counts": plot_species_evidence_counts.__doc__,
    "emergence_timeline": plot_emergence_timeline.__doc__,
    "feature_prevalence": plot_feature_prevalence.__doc__,
    "annual_evidence_counts": plot_annual_evidence_counts.__doc__,
    "taxonomy_composition": plot_taxonomy_composition.__doc__,
    "species_feature_space": plot_species_feature_space.__doc__,
})

In [ ]:
if PARAMS.get("run_phase", False):
    phase5_result = build_phase5_plots(PARAMS)
else:
    phase5_result = {"status": "preflight_only", "reason": "Set run_phase: true to generate Phase 5 figures."}
display(phase5_result)

In [ ]:
plot_dir = resolve_path(PARAMS.get("plot_dir", "data/05_plots/release"))
figure_paths = [
    plot_dir / "species_evidence_counts.png",
    plot_dir / "emergence_timeline.png",
    plot_dir / "feature_prevalence.png",
    plot_dir / "annual_evidence_counts.png",
    plot_dir / "taxonomy_composition.png",
    plot_dir / "species_feature_space.png",
]
display(nbutils.existing_outputs({path.stem: path for path in figure_paths}))
